# Import Libraries & Merge Data

In [1]:
import pandas as pd
import os
import requests

folder_path = "data/2022-citibike-tripdata"
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]

# Empty list to collect each monthly cleaned DataFrame
df_list = []

for file in csv_files:
    print(f"Loading {file}...")
    
    # Read each CSV
    chunk = pd.read_csv(file, low_memory=False)
    
    # Fix datetime for started_at and ended_at
    chunk['started_at'] = pd.to_datetime(chunk['started_at'], errors='coerce')
    chunk['ended_at'] = pd.to_datetime(chunk['ended_at'], errors='coerce')
    
    # Create a 'date' column for merging later
    chunk['date'] = pd.to_datetime(chunk['started_at'].dt.date)
    
    # Collect into list
    df_list.append(chunk)

# Combine all months together
citibike_df = pd.concat(df_list, ignore_index=True)
# Save the full year combined CitiBike dataset
citibike_df.to_csv("citibike_2022_full.csv", index=False)

Loading data/2022-citibike-tripdata\202201-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202201-citibike-tripdata_2.csv...
Loading data/2022-citibike-tripdata\202202-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202202-citibike-tripdata_2.csv...
Loading data/2022-citibike-tripdata\202203-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202203-citibike-tripdata_2.csv...
Loading data/2022-citibike-tripdata\202204-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202204-citibike-tripdata_2.csv...
Loading data/2022-citibike-tripdata\202204-citibike-tripdata_3.csv...
Loading data/2022-citibike-tripdata\202205-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202205-citibike-tripdata_2.csv...
Loading data/2022-citibike-tripdata\202205-citibike-tripdata_3.csv...
Loading data/2022-citibike-tripdata\202206-citibike-tripdata_1.csv...
Loading data/2022-citibike-tripdata\202206-citibike-tripdata_2.csv...
Loading data/2022-ci

This code reads all the CSV files from the 2022 citibike data and merges them all into one large dataframe.

In [8]:
# Replace with your token
token = "wNpPtKCrsJceFkeYikeapoLWDeiZjHni"

headers = {
    'token': token
}

# NOAA API endpoint
url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"

# Parameters
params = {
    "datasetid": "GHCND",
    "stationid": "GHCND:USW00014732",  # LaGuardia Airport
    "startdate": "2022-01-01",
    "enddate": "2022-12-31",
    "limit": 1000,  # 1000 records at a time
    "units": "standard",
    "datatypeid": ["TMAX", "TMIN", "PRCP"],  # Max temp, Min temp, Precip
}

# Request
response = requests.get(url, headers=headers, params=params)

# Check
if response.status_code == 200:
    data = response.json()['results']
    df = pd.DataFrame(data)
    df.to_csv("laguardia_weather_2022.csv", index=False)
    print("Weather data saved to laguardia_weather_2022.csv!")
else:
    print("Error:", response.status_code, response.text)

Weather data saved to laguardia_weather_2022.csv!


Downloaded weather data relevant to my project

In [2]:
# Merge Datasets Together
citibike_df = pd.read_csv("citibike_2022_full.csv", low_memory=False)
weather_df = pd.read_csv("data/laguardia_weather_2022.csv", low_memory=False)

# Fix datetime
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors='coerce')
weather_df['date'] = pd.to_datetime(weather_df['date'], errors='coerce')

# Merge datasets
merged_df = pd.merge(citibike_df, weather_df, on='date', how='left')

# Save merged dataset
merged_df.to_csv("merged_citibike_weather_2022.csv", index=False)

print("✅ Merged and saved merged_citibike_weather_2022.csv!")

✅ Merged and saved merged_citibike_weather_2022.csv!


Creating gitignore to not upload large datasets

In [4]:
!echo "merged_citibike_weather_2022.csv" >> .gitignore

Manually opened "gitignore" to add other large datasets

In [6]:
!git add .
!git commit -m "Add analysis code and exclude large dataset"
!git push origin main

fatal: Out of memory, realloc failed


On branch master
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
  (commit or discard the untracked or modified content in submodules)
	modified:   ../20th-Century (new commits)
	modified:   ../Alice_Network_Analysis (untracked content)
	modified:   ../Test (modified content, untracked content)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/TypicalPancake/20th-Century'
